# PATTERN MINING

## IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
# For visualizations
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

/home/guima/anaconda3/envs/ml/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## CLASSES

In [2]:
class DiabeticDataLoader:
    def __init__(self, df_raw):
        self.df_raw = df_raw.copy()
        self.df_clean = None
        self.df_no_outliers = None
        self.scaler = None

    def clean_data(self):
        df = self.df_raw.copy()

        # Remover colunas constantes
        constant_cols = ['examide', 'citoglipton']
        df.drop(columns=constant_cols, inplace=True, errors='ignore')

        # Substituir '?' por NaN de todas as colunas
        df.replace('?', np.nan, inplace=True)

        # Lidar com max_glu_serum e A1Cresult
        for col in ['max_glu_serum', 'A1Cresult']:
            df[col] = df[col].replace('None', np.nan)
            df[col + '_measured'] = (~df[col].isna()).astype(int)

        # Indicador de peso registrado e conversão
        df['weight_recorded'] = (~df['weight'].isna()).astype(int)
        df['weight'] = df['weight'].astype('category')

        # Colunas categóricas
        categorical_cols = [
            'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id', 'payer_code', 'medical_specialty',
            'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult',
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone',
            'change', 'diabetesMed', 'readmitted'
        ]

        for col in categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')

        self.df_clean = df
        return df

    def remove_outliers(self, df=None):
        if df is None:
            if self.df_clean is None:
                raise ValueError("Data must be cleaned before outlier removal.")
            else:
                df = self.df_clean.copy()

        outlier_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient'
        ]

        self.thresholds = {}
        for col in outlier_cols:
            if col in df.columns:
                # Calcula Min e P99 para ser o thresholds
                low = df[col].min()
                high = df[col].quantile(0.99)
                self.thresholds[col] = (low, high)

        mask = pd.Series(True, index=df.index)
        for feature, (low, high) in self.thresholds.items():
            if feature in df.columns:
                mask &= df[feature].between(low, high)

        df_no_outliers = df.loc[mask].copy()
        self.df_no_outliers = df_no_outliers
        return df_no_outliers

    def get_clean_data(self):
        if self.df_clean is None:
            self.clean_data()
        return self.df_clean

    def get_no_outliers_data(self):
        if self.df_no_outliers is None:
            self.remove_outliers()
        return self.df_no_outliers

    def get_features_target(self):
        df = self.get_no_outliers_data()
        exclude_cols = ['encounter_id', 'patient_nbr', 'readmitted']
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        X = df[feature_cols]
        y = df['readmitted']
        return X, y

    def get_train_test_split(self, test_size=0.2, random_state=42):
        # Limpa os dados de forma global
        df_clean = self.clean_data()

        # Separa X e Y
        X, y = self.get_features_target()

        # Faz o split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=random_state
        )

        # Remove outliers no train
        train_combined = pd.concat([X_train, y_train], axis=1)
        train_combined = self.remove_outliers(train_combined)
        X_train = train_combined.drop(columns=['readmitted'])
        y_train = train_combined['readmitted']

        # Padronização
        numeric_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient',
            'num_procedures', 'number_diagnoses'
        ]

        self.scaler = StandardScaler()
        # FIT apenas no treino
        X_train[numeric_cols] = self.scaler.fit_transform(X_train[numeric_cols])
        # TRANSFORM no teste
        X_test[numeric_cols] = self.scaler.transform(X_test[numeric_cols])

        return X_train, X_test, y_train, y_test

## MINING PROCESS

In [3]:
df_diabetic_data = pd.read_csv(os.path.join('..','data', 'raw','diabetic_data.csv'))
df_diabetic_loader = DiabeticDataLoader(df_diabetic_data)
df_diabetic_clean = df_diabetic_loader.get_clean_data()
df_diabetic_no_outliers = df_diabetic_loader.get_no_outliers_data()

In [4]:
df_diabetic_no_outliers.info()

<class 'pandas.DataFrame'>
Index: 97601 entries, 0 to 101765
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   encounter_id              97601 non-null  int64   
 1   patient_nbr               97601 non-null  int64   
 2   race                      95378 non-null  category
 3   gender                    97601 non-null  category
 4   age                       97601 non-null  category
 5   weight                    2968 non-null   category
 6   admission_type_id         97601 non-null  category
 7   discharge_disposition_id  97601 non-null  category
 8   admission_source_id       97601 non-null  category
 9   time_in_hospital          97601 non-null  int64   
 10  payer_code                58719 non-null  category
 11  medical_specialty         49799 non-null  category
 12  num_lab_procedures        97601 non-null  int64   
 13  num_procedures            97601 non-null  int64   
 14  num_m

In [ ]:
columns_categorical = df_diabetic_no_outliers.astype('category').columns

numerical_cols = df_diabetic_no_outliers.select_dtypes(include=['int64', 'float64']).columns

df_categorical = df_diabetic_no_outliers[columns_categorical].drop(columns=['encounter_id', 'patient_nbr'], errors='ignore')
df_categorical = df_categorical.drop(columns=numerical_cols, errors='ignore') 
df_categorical.columns

Index(['race', 'gender', 'age', 'weight', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'insulin', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [46]:
colunas_medicamentos = [
    'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
    'tolbutamide', 'troglitazone', 'tolazamide', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'A1Cresult', 'metformin', 'repaglinide', 
    'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol'
]

cols_para_alterar = [col for col in colunas_medicamentos if col in df_categorical.columns]

df_categorical[cols_para_alterar] = df_categorical[cols_para_alterar].replace('No', np.nan)

df_transacoes_bool = pd.get_dummies(df_categorical).astype(bool)
df_transacoes_bool

,race_AfricanAmerican,race_Asian,race_Caucasian,race_Hispanic,race_Other,gender_Female,gender_Male,gender_Unknown/Invalid,age_[0-10),age_[10-20),...,insulin_No,insulin_Steady,insulin_Up,change_Ch,change_No,diabetesMed_No,diabetesMed_Yes,readmitted_<30,readmitted_>30,readmitted_NO
0,False,False,True,False,False,True,False,False,True,False,...,True,False,False,False,True,True,False,False,False,True
1,False,False,True,False,False,True,False,False,False,True,...,False,False,True,True,False,False,True,False,True,False
2,True,False,False,False,False,True,False,False,False,False,...,True,False,False,False,True,False,True,False,False,True
3,False,False,True,False,False,False,True,False,False,False,...,False,False,True,True,False,False,True,False,False,True
4,False,False,True,False,False,False,True,False,False,False,...,False,True,False,True,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,True,False,False,False,False,False,True,False,False,False,...,False,False,False,True,False,False,True,False,True,False
101762,True,False,False,False,False,True,False,False,False,False,...,False,True,False,False,True,False,True,False,False,True
101763,False,False,True,False,False,False,True,False,False,False,...,False,False,False,True,False,False,True,False,False,True
101764,False,False,True,False,False,True,False,False,False,False,...,False,False,True,True,False,False,True,False,False,True


In [47]:
itemsets_frequentes = fpgrowth(df_transacoes_bool, min_support=0.05, use_colnames=True) 

In [48]:
itemsets_frequentes.sort_values(by='support', ascending=False).head(30)

,support,itemsets
8,0.767031,frozenset({diabetesMed_Yes})
0,0.747656,frozenset({race_Caucasian})
9,0.594943,frozenset({discharge_disposition_id_1})
44,0.571951,"frozenset({diabetesMed_Yes, race_Caucasian})"
10,0.563560,frozenset({admission_source_id_7})
1,0.545179,frozenset({readmitted_NO})
2,0.544052,frozenset({change_No})
3,0.538304,frozenset({gender_Female})
11,0.531306,frozenset({admission_type_id_1})
613,0.476378,"frozenset({admission_source_id_7, admission_ty..."


In [58]:
regras_filtradas = association_rules(itemsets_frequentes, metric="support", min_threshold=0.05)

regras_filtradas = regras_filtradas[
    (regras_filtradas['lift'] > 1)    
]

p11 = regras_filtradas['support']
p10 = regras_filtradas['antecedent support'] - regras_filtradas['support']
p01 = regras_filtradas['consequent support'] - regras_filtradas['support']
p00 = 1 - regras_filtradas['antecedent support'] - regras_filtradas['consequent support'] + regras_filtradas['support']

regras_finais = regras_filtradas[
    regras_filtradas['consequents'].apply(lambda x: 'readmitted_>30' in x)
]

regras_finais['odds_ratio'] = np.divide((p11 * p00), (p10 * p01))


regras_finais = regras_finais.sort_values(by='odds_ratio', ascending=False)
regras_finais[:5]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,odds_ratio
6669,frozenset({insulin_No}),"frozenset({diabetesMed_No, readmitted_>30})",0.472485,0.070901,0.070901,0.150060,2.116470,1.0,0.037401,1.093134,1.0,0.150060,0.085199,0.575030,inf
6690,frozenset({change_No}),"frozenset({insulin_No, diabetesMed_No, readmit...",0.544052,0.070901,0.070901,0.130320,1.838060,1.0,0.032327,1.068323,1.0,0.130320,0.063954,0.565160,inf
15412,frozenset({diabetesMed_Yes}),"frozenset({change_Ch, readmitted_>30, discharg...",0.767031,0.099722,0.099722,0.130011,1.303728,1.0,0.023232,1.034815,1.0,0.130011,0.033643,0.565005,inf
15470,frozenset({diabetesMed_Yes}),"frozenset({change_Ch, race_Caucasian, readmitt...",0.767031,0.074436,0.074436,0.097044,1.303728,1.0,0.017341,1.025038,1.0,0.097044,0.024426,0.548522,inf
15340,frozenset({diabetesMed_Yes}),"frozenset({change_Ch, race_Caucasian, readmitt...",0.767031,0.128288,0.128288,0.167252,1.303728,1.0,0.029887,1.046790,1.0,0.167252,0.044699,0.583626,inf


In [ ]:
regras = association_rules(itemsets_frequentes, metric="support", min_threshold=0.05)


p11 = regras['support']
p10 = regras['antecedent support'] - regras['support']
p01 = regras['consequent support'] - regras['support']
p00 = 1 - regras['antecedent support'] - regras['consequent support'] + regras['support']

regras = regras[
    regras['consequents'].apply(lambda x: 'readmitted_<30' in x)
]


epsilon = 1e-9 # 0.000000001
regras_finais['odds_ratio'] = ((p11 + epsilon) * (p00 + epsilon)) / ((p10 + epsilon) * (p01 + epsilon))

regras_filtradas = regras_finais[
    (regras_filtradas['lift'] > 1) &
    (regras_filtradas['odds_ratio'] > 1)    
]
regras_finais = regras_finais.sort_values(by='odds_ratio', ascending=False)
regras_finais[:5]

KeyError: 'odds_ratio'

In [60]:
regras_filtradas = association_rules(itemsets_frequentes, metric="support", min_threshold=0.05)

regras_filtradas = regras_filtradas[
    (regras_filtradas['lift'] > 1)    
]

p11 = regras_filtradas['support']
p10 = regras_filtradas['antecedent support'] - regras_filtradas['support']
p01 = regras_filtradas['consequent support'] - regras_filtradas['support']
p00 = 1 - regras_filtradas['antecedent support'] - regras_filtradas['consequent support'] + regras_filtradas['support']

regras_finais = regras_filtradas[
    regras_filtradas['consequents'].apply(lambda x: 'readmitted_NO' in x)
]

regras_finais['odds_ratio'] = np.divide((p11 * p00), (p10 * p01))


regras_finais = regras_finais.sort_values(by='conviction', ascending=False)
regras_finais[:5]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,odds_ratio
6443,"frozenset({diabetesMed_No, discharge_dispositi...","frozenset({insulin_No, change_No, readmitted_NO})",0.136720,0.220182,0.083800,0.612935,2.783762,1.0,0.053697,2.014693,0.742255,0.306847,0.503646,0.496765,8.440091
6541,"frozenset({diabetesMed_No, race_Caucasian, dis...","frozenset({insulin_No, change_No, readmitted_NO})",0.101894,0.220182,0.061844,0.606938,2.756527,1.0,0.039408,1.983957,0.709521,0.237647,0.495957,0.443906,7.214262
7047,"frozenset({diabetesMed_No, gender_Male})","frozenset({insulin_No, change_No, readmitted_NO})",0.104487,0.220182,0.063401,0.606786,2.755835,1.0,0.040395,1.983188,0.711473,0.242667,0.495761,0.447367,7.271082
5503,frozenset({diabetesMed_No}),"frozenset({insulin_No, change_No, readmitted_NO})",0.232969,0.220182,0.140326,0.602340,2.735642,1.0,0.089031,1.961015,0.827157,0.448579,0.490060,0.619830,13.034389
5550,"frozenset({diabetesMed_No, race_Caucasian})","frozenset({insulin_No, change_No, readmitted_NO})",0.175705,0.220182,0.105829,0.602309,2.735504,1.0,0.067142,1.960865,0.769672,0.364853,0.490021,0.541476,9.402593
